In [ ]:
import torch
import torch.nn as nn
import cv2
import numpy as np
from pathlib import Path
from ultralytics import YOLO
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

BASE_PATH = Path('.')
MODEL_PATH = BASE_PATH / 'model_behavioral.pth'
YOLO_MODEL = BASE_PATH / 'yolov8n.pt'

In [ ]:
class CentroidTracker:
    def __init__(self, maxDisappeared=50):
        self.nextObjectID = 0
        self.objects = {}
        self.disappeared = {}
        self.maxDisappeared = maxDisappeared

    def register(self, centroid):
        self.objects[self.nextObjectID] = centroid
        self.disappeared[self.nextObjectID] = 0
        self.nextObjectID += 1

    def deregister(self, objectID):
        del self.objects[objectID]
        del self.disappeared[objectID]

    def update(self, rects):
        if len(rects) == 0:
            for objectID in list(self.disappeared.keys()):
                self.disappeared[objectID] += 1
                if self.disappeared[objectID] > self.maxDisappeared:
                    self.deregister(objectID)
            return self.objects

        input_centroids = np.zeros((len(rects), 2))
        for (i, (startX, startY, endX, endY)) in enumerate(rects):
            cX = int((startX + endX) / 2.0)
            cY = int((startY + endY) / 2.0)
            input_centroids[i] = [cX, cY]

        if len(self.objects) == 0:
            for i in range(0, len(input_centroids)):
                self.register(input_centroids[i])
        else:
            objectIDs = list(self.objects.keys())
            objectCentroids = list(self.objects.values())
            D = np.zeros((len(objectCentroids), len(input_centroids)))

            for i in range(len(objectCentroids)):
                for j in range(len(input_centroids)):
                    D[i, j] = np.linalg.norm(objectCentroids[i] - input_centroids[j])

            rows = D.min(axis=1).argsort()
            cols = D.argmin(axis=1)[rows]
            used_rows = set()
            used_cols = set()

            for (row, col) in zip(rows, cols):
                if row in used_rows or col in used_cols:
                    continue
                if D[row, col] > 50:
                    continue
                objectID = objectIDs[row]
                self.objects[objectID] = input_centroids[col]
                self.disappeared[objectID] = 0
                used_rows.add(row)
                used_cols.add(col)

            unused_rows = set(range(0, len(objectCentroids))).difference(used_rows)
            unused_cols = set(range(0, len(input_centroids))).difference(used_cols)

            if len(objectCentroids) >= len(input_centroids):
                for row in unused_rows:
                    objectID = objectIDs[row]
                    self.disappeared[objectID] += 1
                    if self.disappeared[objectID] > self.maxDisappeared:
                        self.deregister(objectID)
            else:
                for col in unused_cols:
                    self.register(input_centroids[col])

        return self.objects

print('CentroidTracker defined')

In [ ]:
class BehavioralLSTM(nn.Module):
    def __init__(self, input_size=10, hidden_size=32, num_layers=1, dropout=0.5):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, dropout=0.0 if num_layers == 1 else dropout, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(hidden_size, 16)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(16, 2)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_out = lstm_out[:, -1, :]
        x = self.dropout(last_out)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

print('BehavioralLSTM model defined')

In [ ]:
model = BehavioralLSTM(input_size=10, hidden_size=32, num_layers=1, dropout=0.5).to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()

print(f'Model loaded from: {MODEL_PATH}')

yolo_model = YOLO(str(YOLO_MODEL))
yolo_model.to(device)
print(f'YOLO model loaded')

In [ ]:
def compute_sequence_features(person_trajectory, frame_h, frame_w, seq_len=18):
    if len(person_trajectory) < seq_len:
        return None
    
    recent_frames = person_trajectory[-seq_len:]
    features = np.array(recent_frames, dtype=np.float32)
    return features.reshape(1, seq_len, 10)

def get_risk_level(risk_score):
    if risk_score < 0.33:
        return 'LOW', (0, 255, 0)
    elif risk_score < 0.66:
        return 'MEDIUM', (0, 165, 255)
    else:
        return 'HIGH', (0, 0, 255)

print('Inference functions defined')

In [ ]:
video_path = BASE_PATH / 'Footage of jewellery store robbery.mp4'

if not video_path.exists():
    print(f'Video not found: {video_path}')
    print('Place your video in the current directory')
else:
    cap = cv2.VideoCapture(str(video_path))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    output_path = BASE_PATH / f'{video_path.stem}_risk_analysis.mp4'
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(str(output_path), fourcc, fps, (frame_width, frame_height))
    
    tracker = CentroidTracker(maxDisappeared=50)
    person_trajectories = {}
    person_features = {}
    
    frame_idx = 0
    frame_skip = max(1, fps // 5)
    high_risk_count = 0
    total_persons = 0
    
    print(f'Processing video: {video_path.name}')
    print(f'  Resolution: {frame_width}x{frame_height}')
    print(f'  FPS: {fps}, Total frames: {frame_count}')
    print(f'  Frame skip: {frame_skip}')
    print()
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        frame_idx += 1
        
        frame_resized = cv2.resize(frame, (640, 480))
        frame_h, frame_w = frame_resized.shape[:2]
        
        results = yolo_model.predict(frame_resized, conf=0.4, verbose=False)
        rects = []
        
        if results[0].boxes is not None:
            for detection in results[0].boxes:
                x1, y1, x2, y2 = map(int, detection.xyxy[0].cpu().numpy())
                conf = float(detection.conf[0].cpu().numpy())
                cls = int(detection.cls[0].cpu().numpy())
                
                if conf > 0.4 and cls == 0:
                    rects.append((x1, y1, x2, y2))
        
        objects = tracker.update(rects)
        
        for (objectID, centroid) in objects.items():
            if objectID not in person_trajectories:
                person_trajectories[objectID] = []
                person_features[objectID] = []
            
            for (x1, y1, x2, y2) in rects:
                cx = int((x1 + x2) / 2.0)
                cy = int((y1 + y2) / 2.0)
                if np.allclose([cx, cy], centroid, atol=5):
                    norm_x = cx / (frame_w + 1e-6)
                    norm_y = cy / (frame_h + 1e-6)
                    
                    if len(person_trajectories[objectID]) > 0:
                        prev_x = person_trajectories[objectID][-1][0]
                        prev_y = person_trajectories[objectID][-1][1]
                        dx = norm_x - prev_x
                        dy = norm_y - prev_y
                    else:
                        dx = dy = 0.0
                    
                    speed = np.sqrt(dx**2 + dy**2)
                    theta = np.arctan2(dy, dx)
                    
                    if len(person_trajectories[objectID]) > 0:
                        prev_feat = person_trajectories[objectID][-1]
                        prev_speed = prev_feat[4]
                        prev_theta = prev_feat[5]
                        delta_theta = theta - prev_theta
                        acc = speed - prev_speed
                    else:
                        delta_theta = acc = 0.0
                    
                    movement_variance = 0.0
                    path_efficiency = 0.0
                    
                    feature_vec = [norm_x, norm_y, dx, dy, speed, theta, delta_theta, acc, movement_variance, path_efficiency]
                    person_trajectories[objectID].append(feature_vec)
                    
                    risk_score = 0.5
                    
                    if len(person_trajectories[objectID]) >= 18:
                        seq = np.array(person_trajectories[objectID][-18:], dtype=np.float32).reshape(1, 18, 10)
                        with torch.no_grad():
                            seq_tensor = torch.tensor(seq, dtype=torch.float32).to(device)
                            output = model(seq_tensor)
                            prob = torch.softmax(output, dim=1)[0, 1].item()
                            risk_score = prob
                    
                    risk_level, color = get_risk_level(risk_score)
                    
                    scale_x = frame_width / frame_w
                    scale_y = frame_height / frame_h
                    x1_orig = int(x1 * scale_x)
                    y1_orig = int(y1 * scale_y)
                    x2_orig = int(x2 * scale_x)
                    y2_orig = int(y2 * scale_y)
                    
                    cv2.rectangle(frame, (x1_orig, y1_orig), (x2_orig, y2_orig), color, 2)
                    cv2.putText(frame, f'ID:{objectID} Risk:{risk_score:.2f}', (x1_orig, y1_orig-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
                    cv2.putText(frame, risk_level, (x1_orig, y1_orig-30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
                    
                    if risk_score > 0.66:
                        high_risk_count += 1
                    
                    total_persons += 1
        
        cv2.putText(frame, f'Frame: {frame_idx}/{frame_count}', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.putText(frame, f'High Risk: {high_risk_count}', (10, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
        
        writer.write(frame)
        
        if frame_idx % 100 == 0:
            print(f'  Processed {frame_idx}/{frame_count} frames')
    
    cap.release()
    writer.release()
    
    print()
    print('='*70)
    print('ANALYSIS COMPLETE')
    print('='*70)
    print(f'Output video saved: {output_path}')
    print(f'Total frames processed: {frame_idx}')
    print(f'Total persons detected: {total_persons}')
    print(f'High-risk detections: {high_risk_count}')